# Graphics Accuracy Evaluator

**The Graphics Accuracy Evaluator** checks whether a math graphic (plot, chart, diagram, equation, or worked solution) correctly shows what its accompanying claim states, and whether the image is itself sound: complete, legible, and internally consistent. The claim is trusted; the image is under review.

**Inputs**
- `image_paths`: a one-item list with the local path of the image (PNG, JPEG or WebP, checked against the bounds in `input_schema.json` before any model call).
- `claim`: a statement about the image, or a question with its expected answer written as `Question: "…" The answer is ….`

**Output** (`output_schema.json`): `is_correct`; `reasoning`, ending in "The image yields X; the specification states Y; therefore is_correct = ___"; `observed`; `defects`; `errors`; `correction`; and `basis`:

| `basis` | `is_correct` | meaning |
| --- | --- | --- |
| `supported` | true | the derivation completed and matches the claim |
| `contradicted` | false | the derivation completed and disagrees with the claim |
| `unverified` | false | the derivation could not be completed from the image |
| `defective` | false | the image itself is flawed, whatever the claim |

This notebook loads `config.json`, the prompts and the schemas from this directory, runs one example, and runs every case in `fixtures.json`. Early access.

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic python-dotenv Pillow


In [ ]:
import base64
import getpass
import hashlib
import io
import json
import os
import pprint as pp
from pathlib import Path

from PIL import Image
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [ ]:
ASSETS_DIR = Path(".")
with open(ASSETS_DIR / "config.json") as f:
    CONFIG = json.load(f)
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

STEP = CONFIG["steps"][0]

# Load every prompt message declared in config and verify its sha256 against the declared hash.
PROMPTS = {}
for msg_spec in STEP["prompt"]["messages"]:
    raw = (ASSETS_DIR / msg_spec["source_path"]).read_bytes()
    actual_sha = hashlib.sha256(raw).hexdigest()
    assert actual_sha == msg_spec["sha256"], (
        f"prompt drift detected for {msg_spec['source_path']}: "
        f"declared {msg_spec['sha256'][:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPTS[msg_spec["role"]] = raw.decode("utf-8")

print(f"Loaded {CONFIG['evaluator']['id']} from {ASSETS_DIR.resolve()}")
print(f"  model:       {STEP['model']['name']}")
print(f"  temperature: {STEP['generation']['temperature']}  (null: not sent)")
print(f"  prompts:     {', '.join(m['source_path'] for m in STEP['prompt']['messages'])}")

In [ ]:
# -------------------------------------------------------------------------
# Image input: the image_paths attachment declared in config
# -------------------------------------------------------------------------
# The bounds input_schema.json declares for image_paths (its count, and the x-image
# block on each file), checked before any model call. An image outside them is
# rejected, not resized. The format is read from the file's first bytes, not its name.
_IMAGES = INPUT_SCHEMA["properties"]["image_paths"]
_BOUNDS = _IMAGES["items"]["x-image"]
_SIGNATURES = [
    (lambda b: b.startswith(b"\x89PNG\r\n\x1a\n"), "image/png"),
    (lambda b: b.startswith(b"\xff\xd8\xff"), "image/jpeg"),
    (lambda b: b[:4] == b"RIFF" and b[8:12] == b"WEBP", "image/webp"),
]


def load_image(reference):
    """Read one local image (relative to ASSETS_DIR, or absolute), enforce the x-image bounds, return a data URL."""
    path = Path(reference).expanduser()
    data = (path if path.is_absolute() else ASSETS_DIR / path).read_bytes()
    if not _BOUNDS["min_bytes"] <= len(data) <= _BOUNDS["max_bytes"]:
        raise ValueError(f"{reference!r} is {len(data):,} bytes; allowed {_BOUNDS['min_bytes']:,} to {_BOUNDS['max_bytes']:,}")
    mime = next((m for test, m in _SIGNATURES if test(data)), None)
    if mime not in _BOUNDS["formats"]:
        raise ValueError(f"{reference!r} is not an accepted image; allowed {', '.join(_BOUNDS['formats'])}")
    width, height = Image.open(io.BytesIO(data)).size
    if min(width, height) < _BOUNDS["min_edge"] or max(width, height) > _BOUNDS["max_edge"]:
        raise ValueError(f"{reference!r} is {width}x{height} px; each side must be {_BOUNDS['min_edge']} to {_BOUNDS['max_edge']} px")
    return f"data:{mime};base64,{base64.b64encode(data).decode('ascii')}"


def compose_claim(question, answer):
    """The claim text for a question with an expected answer."""
    return f'Question: "{question.strip()}" The answer is {answer.strip()}.'


# -------------------------------------------------------------------------
# Model: structured output against output_schema.json
# -------------------------------------------------------------------------
_REQUEST_SCHEMA = {k: v for k, v in OUTPUT_SCHEMA.items() if k not in ("$schema", "$id")}
llm = ChatGoogleGenerativeAI(
    model=STEP["model"]["name"],
    temperature=STEP["generation"]["temperature"],  # None: no temperature is sent
)
structured_llm = llm.with_structured_output(_REQUEST_SCHEMA, method="json_schema", include_raw=True)


def evaluate_graphics_accuracy(image_paths, claim):
    """Evaluate one image against one claim. Returns the parsed output and token usage."""
    if not _IMAGES["minItems"] <= len(image_paths) <= _IMAGES["maxItems"]:
        raise ValueError(f"{len(image_paths)} images; allowed {_IMAGES['minItems']} to {_IMAGES['maxItems']}")
    images = [{"type": "image_url", "image_url": {"url": load_image(p)}} for p in image_paths]
    user_text = PROMPTS["user"].replace("{claim}", claim)
    # The image goes ahead of the user text, as the step's attachments declare.
    messages = [SystemMessage(content=PROMPTS["system"]), HumanMessage(content=[*images, {"type": "text", "text": user_text}])]
    response = structured_llm.invoke(messages)
    if response.get("parsing_error") or response.get("parsed") is None:
        raise ValueError(f"output failed the schema: {response.get('parsing_error')}")
    return {"output": response["parsed"], "usage": response["raw"].usage_metadata}

In [ ]:
# One case from fixtures.json: a question with its expected answer, on a counting picture.
result = evaluate_graphics_accuracy(
    image_paths=["images/apples-in-baskets.png"],
    claim=compose_claim("How many apples are in the three baskets altogether?", "12"),
)
pp.pprint(result["output"])
print("usage:", result["usage"])

In [ ]:
# -------------------------------------------------------------------------
# Fixtures: run every case and compare with its expected verdict
# -------------------------------------------------------------------------
# Each case pins is_correct and, where stable, basis. One run per case: a single
# call can land on the other side of a borderline item, so treat a miss as a
# signal to re-run, not a measurement.
with open(ASSETS_DIR / CONFIG["fixtures"]["path"]) as f:
    fixtures = json.load(f)

passed = 0
for case in fixtures:
    expected = case["expected"]
    try:
        output = evaluate_graphics_accuracy(**case["input"])["output"]
    except ValueError as err:
        print(f"ERROR  {case['id']}: {err}")
        continue
    ok = output["is_correct"] == expected["is_correct"] and expected.get("basis", output["basis"]) == output["basis"]
    passed += ok
    want = f"{expected['is_correct']}/{expected.get('basis', '*')}"
    print(f"{'PASS' if ok else 'FAIL'}  {case['id']:<26} expected {want:<18} got {output['is_correct']}/{output['basis']}")
print(f"\n{passed}/{len(fixtures)} fixtures passed")